In [33]:
import os
import glob
import json
import time
import arcgis
from arcgis.gis import GIS
from arcgis.gis import Item

### Setup Environmental Variables

In [4]:
arcgis.env.verbose=True

In [5]:
profile = 'your_enterprise_profile'
#portalUrl = "https://sha-imgcl-d01.esri.com/portal"
#portalUN = "admin"
#portalPW = "adminadmin"
imageFolderPath = r"./BU"
imageSuffix = ".jpg"
#arcgis.env.verbose =  False

In [6]:
gis = GIS(profile=profile, verify_cert=False)
gis

GIS @ https://sha-imgcl-d01.esri.com/portal version:7.3

In [8]:
from arcgis.raster import analytics
isSupport = analytics.is_supported(gis)

if isSupport:
    print("Your enterprise gis supports orthomapping service")
else:
    print("Your enterprise gis does not support orthomapping service")
    exit()

Your enterprise gis supports orthomapping service


### Check if Tool Creates

- This drives the whole testing documentation

In [11]:
from arcgis._impl.tools import _OrthoMappingTools
omt = gis._tools.orthomapping
assert isinstance(omt, _OrthoMappingTools)

### Create Image Collection for Testing

In [21]:
def currentTime():
    return time.strftime("%Y%m%d%H%M%S", time.localtime())
prjFolderName = "omProject" + currentTime()
gis.content.create_folder(folder=prjFolderName, owner=gis.users.me.username)
prjFolderName

'omProject20190906071843'

In [22]:
# Add Images and Create Image Collection
username = gis.users.me.username
imageList = glob.glob(os.path.join(imageFolderPath, '*.JPG'), recursive=True)
imageItemList = []
itemPropTemplate = {"type": "Image"}

for imageFullPath in imageList:
    imageName = imageFullPath[imageFullPath.rfind("\\")+1:]
    itemPropTemplate["title"] = imageName
    itemPropTemplate["tags"] = imageName
    itemPropTemplate["description"] = imageName

    imageItem = gis.content.add(item_properties=itemPropTemplate, data=imageFullPath,
                                owner=username, folder=prjFolderName)
    imageItemList.append(imageItem)

In [23]:
swRasterTypeParams = {"gps": [['YUN_0040.JPG', 34.0069887, -117.09279029999999],
  ['YUN_0041.JPG', 34.0070131, -117.09311519972222],
  ['YUN_0042.JPG', 34.0070381, -117.09346329972222],
  ['YUN_0043.JPG', 34.00706339972222, -117.09381479999999],
  ['YUN_0044.JPG', 34.0070879, -117.09416449999999],
  ['YUN_0045.JPG', 34.007113099722226, -117.09450929972222],
  ['YUN_0046.JPG', 34.0071384, -117.09485779972222],
  ['YUN_0076.JPG', 34.00668639972222, -117.09463709972222],
  ['YUN_0077.JPG', 34.00666089972222, -117.09428809972222],
  ['YUN_0078.JPG', 34.00663709972222, -117.09395939999999],
  ['YUN_0079.JPG', 34.0066113, -117.09360669972222],
  ['YUN_0080.JPG', 34.00658549972222, -117.09325299999999],
  ['YUN_0081.JPG', 34.0065606, -117.0929043]],
"cameraProperties":{"maker":"Yuneec","model":"E90","focallength":8,"columns":5472,"rows":3648,"pixelsize":0.0024},
"isAltitudeFlightHeight":"false",
"averagezdem": {"url": "https://rais.dev.geocloud.com/arcgis/rest/services/Hosted/WorldSRTM90m/ImageServer"}}

In [24]:
from arcgis.raster.analytics import create_image_collection
image_collection_name = "imgcollect" + currentTime()
image_collection = create_image_collection(image_collection=image_collection_name,
                                           input_rasters=imageItemList,
                                           raster_type_name="UAV/UAS",
                                           raster_type_params=swRasterTypeParams,
                                           out_sr=32632,
                                           gis=gis)

Submitted.
Executing...
Start Time: Friday, September 6, 2019 3:31:20 PM
Running script CreateImageCollection...
New portal item created, ID: 09e596ab2e2d476db176272bf1891c41
Output item id is: 09e596ab2e2d476db176272bf1891c41
Output image service url is: https://SHA-IMGCL-D01.esri.com:6443/arcgis/rest/services/Hosted/imgcollect20190906073101/ImageServer
Input raster is: ['116029a0b56f4040a1a46af16f668e5b', '0e2ababc326d4b71aa5430b62945a72c', 'ed662823895f439aaa3cfaaaad51e5ee', '290a432ed4b8429e9b2ab14c09c3a078', '81285ebefa1145ebaaf1b60e25bcf261', '42863b8681194922bc984f1d3ed3b484', '190ecf923783436db8aa53acf851c20c', '9da8c5fd15a04ff0b9cf5498c3a464ae', '3725419a473a460d8b6b3102bcef65c0', '68b59663b7f0482db03e14607f092bcd', 'c990daa10c6746dea5c23874d6b28ca8', '55a13f5565d94373be912f225ab29e54', 'd3518cd4faf546ddbcc5010ee3c74043']
Raster type JSON is: {'rasterTypeName': 'UAV/UAS', 'rasterTypeParameters': {'gps': [['YUN_0040.JPG', 34.0069887, -117.09279029999999], ['YUN_0041.JPG', 34.00

### Alter Processing States


In [25]:
job_aps = omt.alter_processing_states(image_collection=image_collection, 
                                      new_states={"blockadjustment": "raw","dem": "Dense_Natual_Neighbor","seamlines":"VORONOI","colorcorrection":"SingleColor"}, 
                                      gis=gis, future=True)
assert job_aps
assert job_aps.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:14:50 PM
Running script AlterProcessingStates...
Setting image collection orthomapping states
Successfully set new orthomapping states.
AlterProcessingStates GP Job: j5f1160c9b6ff472385747b222dd604cf finished successfully.


In [26]:
job_aps.result()

{'orthomapping': {'blockadjustment': 'raw',
  'dem': 'Dense_Natual_Neighbor',
  'seamlines': 'VORONOI',
  'colorcorrection': 'SingleColor'}}

### Compute Color Correction

In [27]:
ccc_job = omt.compute_color_correction(image_collection,
                                    color_correction_method="DODGING",
                                    dodging_surface="SECOND_ORDER",
                                    context={"skipRows": 10, "skipCols": 10, "reCalculateStats": "OVERWRITE"},
                                    gis = gis,
                                    future=True)
assert ccc_job
assert ccc_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:15:12 PM
Running script ComputeColorCorrection...
Mosaic candidates for image collections are not computed.
Calculate statistics for image collection.
Computing color correction...
ComputeColorCorrection GP Job: j0a3efb7c0d9643668b9008a8597f2cc2 finished successfully.


### Compute Control Points

In [28]:
job_ccp = omt.compute_control_points(image_collection,
                                     reference_image="https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
                                     image_location_accuracy='Low',
                                     gis=gis,
                                     future=True)
assert job_ccp
assert job_ccp.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:16:02 PM
Running script ComputeControlPoints...
Computing control points for image collection based on reference layer URL: https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer
New gound control points and tie points computed.
Creating new gound control points sets.
Succeeded at Friday, September 6, 2019 4:21:24 PM (Elapsed Time: 5 minutes 22 seconds)
ComputeControlPoints GP Job: jfa0cfd500d254e5eafb867033d46f796 finished successfully.


### Compute Seamlines

In [29]:
context={"minRegionSize":100,"blendType":"Both","blendWidth":None,
         "blendUnit":"Pixels","requestSizeType":"Pixels",
         "requestSize":1000,"minThinnessRatio":0.05,"maxSilverSize":20}
cs_job = omt.compute_seamlines(image_collection=image_collection,
                      seamlines_method="DISPARITY",
                      context=context,
                      gis=gis,
                      future=True)
assert cs_job
assert cs_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:21:33 PM
Running script ComputeSeamlines...
Mosaic candidates for image collections are not computed.
Computing seamlines with Disparity method...
Finished computing seamlines with Disparity method.
Getting image service info...
ComputeSeamlines GP Job: j06fd1dade5104a859d1192b0904f06ed finished successfully.


### Compute Sensor Model

In [30]:
csm_job = omt.compute_sensor_model(image_collection, mode='QUICK', location_accuracy="Medium", future=True)
assert csm_job
assert csm_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:22:52 PM
Running script ComputeSensorModel...
Computing camera model and adjustment in "Quick" Mode for image collections...
Image collections adjusted.
Mosaic candidates for image collections are not computed.
ComputeSensorModel GP Job: j3f766d1a20074c6b8dd9f327931f52ad finished successfully.


### Edit Control Points

In [31]:
inputControlPoints = [{"status":1,"type":2,"gcpid":"GCP9","tag":"GCP9","x":-117.0926538,
                      "y":34.00704253,"z":634.2175,"mapx":-13034694.596648848,
                      "mapy":4029747.7050537546,"spatialReference":{"wkid":4326},
                      "id":1,"pointId":1,"xyAccuracy":"0.008602325","zAccuracy":"0.015",
                      "imagePoints":[{"imageID":12,"x":3458.3502468937245,"y":-4424.879721170419,
                                      "u":562.4306077298716,"v":-124.96945105501322},
                                     {"imageID":1,"x":2453.4233514297666,"y":-2001.2854377410533,
                                      "u":3056.8887978139765,"v":-1908.7056791900693},
                                     {"imageID":2,"x":1784.7197170631089,"y":-1114.535011913254,
                                      "u":3058.6565174539687,"v":-2960.846400950005},
                                     {"imageID":13,"x":4672.717414590265,"y":-2871.3511238888987,
                                      "u":617.2496070314755,"v":-1243.7106361762521}],"nlinks":4,"hasphoto":True},
                     {"status":1,"type":2,"gcpid":"GCP26","tag":"GCP26","x":-117.0934208,"y":34.00643003,"z":633.5438,
                      "mapx":-13034779.978698289,"mapy":4029665.454745273,"spatialReference":{"wkid":4326},
                      "id":5,"pointId":5,"xyAccuracy":"0.00781025","zAccuracy":"0.015",
                      "imagePoints":[{"imageID":2,"x":5004.379068578275,"y":-1272.9293022678344,
                                      "u":563.4855594058413,"v":-867.580137100088}],
                      "nlinks":1,"hasphoto":True}]


In [32]:
ecp_job = omt.edit_control_points(image_collection=image_collection,
                        input_control_points=inputControlPoints, 
                       gis=gis, future=True)
assert ecp_job
assert ecp_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:25:10 PM
Running script EditControlPoints...
EditControlPoints GP Job: j9f0a5d9ed4cd4767981139f77e8a955c finished successfully.


### Generate DEM

In [34]:
gen_dem_job = omt.generate_dem(image_collection=image_collection, output_name=None, 
                               cell_size=0.1314245599999937,
                               surface_type="DSM", 
                               matching_method="ETM",
                               context={"maxObjectSize":15,"minAngle":10,"maxAngle":70,"minOverlap":0.6,"maxGSDDif":2,"numImagePairs":8,"adjQualityThreshold":0.2,"method":"TRIANGULATION","smoothingMethod":"GAUSS5x5","applyToOrtho":False,"regenPointCloud":False}, 
                               gis=gis,
                              future=True)
assert gen_dem_job
assert isinstancestance(gen_dem_job.result(), Item)
assert gen_dem_job.result().delete()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:25:40 PM
Running script GenerateDEM...
Generating stereo pairs...
Stereo pairs generated.
Generating point clouds...
Point clouds generated.
Generating DEM...
DEM generated.
Updating service with DEM path...
Getting image service info...
Updating service: https://SHA-IMGCL-D01.esri.com:6443/arcgis/admin/services/Hosted/Generate_DEM_FIV7YQ.ImageServer/edit
Succeeded at Friday, September 6, 2019 4:27:39 PM (Elapsed Time: 1 minutes 58 seconds)
GenerateDEM GP Job: j83b44800576640ae9c6f4009912a549c finished successfully.


### Generate Ortho Mosaic

In [35]:
orthoMosaicName = "OMc" + currentTime() + "c"
othmos_job = omt.generate_orthomosaic(image_collection,
                         output_name=orthoMosaicName,
                         regen_seamlines=False, recompute_color_correction=False,
                         context=None,
                         gis=gis,
                         future=True)
assert othmos_job
res = othmos_job.result()
assert res
assert isinstance(res, Item)
assert res.delete()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:28:24 PM
Running script GenerateOrthomosaic...
Updating service with orthomosaic dataset path...
Getting image service info...
Completed script GenerateOrthomosaic...
Succeeded at Friday, September 6, 2019 4:29:00 PM (Elapsed Time: 35.90 seconds)
GenerateOrthomosaic GP Job: je97bf2c118f74d58b38fc91222b24e7d finished successfully.


### Generate Report

In [36]:
report_link_job = omt.generate_report(image_collection=image_collection, gis=gis, future=True)
assert report_link_job
assert report_link_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:29:25 PM
Running script GenerateReport...
GenerateReport GP Job: j35ded871b718414a8fb2e7f6d8b7d9b5 finished successfully.


### Get Processing States

In [37]:
job_gps = omt.get_processing_states(image_collection = image_collection, future=True)
assert job_gps.result()

Start Time: Friday, September 6, 2019 4:29:43 PM
Running script GetProcessingStates...
Completed script GetProcessingStates...
Succeeded at Friday, September 6, 2019 4:29:45 PM (Elapsed Time: 1.49 seconds)
GetProcessingStates GP Job: ja01f17830600404082365ddd5dba8a60 finished successfully.


### Match Control Points

In [38]:
inputControlPoints= [{"pointId":5,"x":-117.0934208,"y":34.00643003,"z":633.5438,
                      "xyAccuracy":"0.00781025","zAccuracy":"0.015",
                      "imagePoints":[{"imageID":2,"x":5022.736523387883,"y":-1267.6047690326218}]}]
mcp_job = omt.match_control_points(image_collection=image_collection, 
                                   control_points=inputControlPoints,
                                   gis=gis, future=True)
assert mcp_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:29:53 PM
Running script MatchControlPoints...
control points JSON: [{'pointId': 5, 'x': -117.0934208, 'y': 34.00643003, 'z': 633.5438, 'xyAccuracy': '0.00781025', 'zAccuracy': '0.015', 'imagePoints': [{'imageID': 2, 'x': 5022.736523387883, 'y': -1267.6047690326218}]}]
GCP spatial reference from context: 
Image point spatial reference from context: ICS
GCP spatial reference used: "PROJCS['WGS_1984_UTM_Zone_11N',GEOGCS['GCS_WGS_1984',DATUM['D_WGS_1984',SPHEROID['WGS_1984',6378137.0,298.257223563]],PRIMEM['Greenwich',0.0],UNIT['Degree',0.0174532925199433]],PROJECTION['Transverse_Mercator'],PARAMETER['False_Easting',500000.0],PARAMETER['False_Northing',0.0],PARAMETER['Central_Meridian',-117.0],PARAMETER['Scale_Factor',0.9996],PARAMETER['Latitude_Of_Origin',0.0],UNIT['Meter',1.0]];-5120900 -9998100 10000;-100000 10000;-100000 10000;0.001;0.001;0.001;IsHighPrecision" 
Image point spatial reference used: ICS
MatchControlPoints GP

### Query Camera Info

In [39]:
import pandas as pd
qci_job = omt.query_camera_info(query=None, future=True)
res = qci_isinstanceresult()
assert isinstance(res, pd.DataFrame)

QueryCameraInfo GP Job: j44f45e734e1e47b2b900e02ebd637344 finished successfully.


### Query Control Points

In [40]:
## Setup the process
ccp_job = omt.compute_control_points(image_collection=image_collection,
                       reference_image="https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
                       image_location_accuracy='Low',
                       gis=gis, future=True)
ccp_job.result()
cp_job = omt.query_control_points(image_collection=image_collection.url,where="pointID>=0", future=True)
assert cp_job
assert cp_job.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:30:19 PM
Running script ComputeControlPoints...
Computing control points for image collection based on reference layer URL: https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer
Added new gound control points sets.
Completed script ComputeControlPoints...
Succeeded at Friday, September 6, 2019 4:35:04 PM (Elapsed Time: 4 minutes 45 seconds)
ComputeControlPoints GP Job: j26603db6a22546ba95f0102f44d8dacb finished successfully.
Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:35:11 PM
Running script QueryControlPoints...
Control point table is c:\rasterstore\imgcollect20190906073101\imgcollect20190906073101.gdb\imgcollect20190906073101_1_p
QueryControlPoints GP Job: j3a15965c177548be96169dde86c9b93b finished successfully.


### Reset Image collection

In [41]:
reset_flag = omt.reset_image_collection(image_collection=image_collection, gis=gis, future=True)
assert reset_flag.result()

Submitted.
Executing...
Start Time: Friday, September 6, 2019 4:35:27 PM
Running script ResetImageCollection...
Input image collection is: https://SHA-IMGCL-D01.esri.com:6443/arcgis/rest/services/Hosted/imgcollect20190906073101/ImageServer
Service admin URL: {}
Resetting adjustment...
Done resetting adjustment.
Rebuilding footprints...
Done rebuilding footprints.
Resetting seamlines...
Done resetting seamlines.
Resetting orthomapping metadata...
Succeeded at Friday, September 6, 2019 4:36:13 PM (Elapsed Time: 45.72 seconds)
ResetImageCollection GP Job: jcd1107c6b773478f8d18f4d411e2b843 finished successfully.


### Clean Up

In [44]:
image_collection.delete()
for img in imageItemList:
    img.delete()

In [45]:
gis.content.delete_folder(prjFolderName)

True